In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# ── Feature Engineering ────────────────────────────────────────
# 1. Extract Title FIRST (before filling Age)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don',
                                    'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})

# 2. Fill Age BEFORE creating AgeGroup
df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
df['Age'] = df['Age'].fillna(df['Age'].median())  # safety net

# 3. Now create AgeGroup safely
df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,35,60,100],
                         labels=[0, 1, 2, 3, 4])  # use numbers directly
df['AgeGroup'] = df['AgeGroup'].astype(float)

# 4. Family features
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 5. Safe FarePerPerson
df['FarePerPerson'] = df['Fare'] / df['FamilySize'].replace(0, 1)

# 6. Cabin known or not (77% missing — don't use the value, just flag it)
df['HasCabin'] = df['Cabin'].notnull().astype(int)

# 7. Encode categories
le = LabelEncoder()
df['Sex_encoded'] = le.fit_transform(df['Sex'])
df['Title_encoded'] = le.fit_transform(df['Title'])
df['Embarked'] = df['Embarked'].fillna('S')
df['Embarked_encoded'] = le.fit_transform(df['Embarked'])

# ── Compare raw vs engineered ──────────────────────────────────
raw_features = ['Pclass', 'Sex_encoded', 'Age', 'Fare', 'SibSp', 'Parch']
eng_features = ['Pclass', 'Sex_encoded', 'Age', 'AgeGroup', 'FamilySize',
                'IsAlone', 'FarePerPerson', 'HasCabin',
                'Title_encoded', 'Embarked_encoded']

model = RandomForestClassifier(n_estimators=100, random_state=42)

X_raw = df[raw_features].fillna(0)
X_eng = df[eng_features].fillna(0)
y = df['Survived']

score_raw = cross_val_score(model, X_raw, y, cv=5, scoring='accuracy').mean()
score_eng = cross_val_score(model, X_eng, y, cv=5, scoring='accuracy').mean()

print(f"Raw features:        {score_raw:.4f}")
print(f"Engineered features: {score_eng:.4f}")
print(f"Improvement:         +{(score_eng - score_raw):.4f}")

Raw features:        0.8081
Engineered features: 0.8103
Improvement:         +0.0022


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_raw = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

lr_eng = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

lr_score_raw = cross_val_score(lr_raw, X_raw, y, cv=5, scoring='accuracy').mean()
lr_score_eng = cross_val_score(lr_eng, X_eng, y, cv=5, scoring='accuracy').mean()

print(f"LogReg - Raw features:        {lr_score_raw:.4f}")
print(f"LogReg - Engineered features: {lr_score_eng:.4f}")
print(f"Improvement:                  +{(lr_score_eng - lr_score_raw):.4f}")

LogReg - Raw features:        0.7890
LogReg - Engineered features: 0.7946
Improvement:                  +0.0056
